# Tutorial 1: Setup & First Run

**Time:** 20 minutes | **Level:** Beginner | **Requirements:** Python 3.10+, 4 GB RAM

In [ ]:
import os, sys
# Set working directory to repo root
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '../../..'))
os.chdir(REPO_ROOT)
sys.path.insert(0, REPO_ROOT)
print(f'Working directory: {os.getcwd()}')

## What You'll Learn

- Install the FL platform and its dependencies
- Verify your environment is ready
- Train a simple model and run inference
- Run the test suite

## Step 1: Clone and Install

```bash
git clone https://github.com/govtech-data-practice/Fl_deployment.git fl-reference
cd fl-reference

# Create a virtual environment
python -m venv .venv
source .venv/bin/activate  # Linux/macOS
# .venv\Scripts\activate   # Windows

# Install core + dev dependencies
pip install -e ".[dev]"
```

**Checkpoint:** `pip list | grep flwr` should show Flower installed.

## Step 2: Verify Your Environment

In [ ]:
!python3 -c "import flwr; print('Flower', flwr.__version__)"
!python3 -c "import torch; print('PyTorch', torch.__version__, '| CUDA:', torch.cuda.is_available())"
!python3 -c "import fl_pets; print('fl_pets: ok')"

**Expected output:**

```
Flower 1.30.0
PyTorch 2.x.x | CUDA: True (or False on CPU)
fl_pets: ok
```

## Step 3: Load Data

The repo includes a **25,000-record sample** of real credit card transactions from the [Kaggle Credit Card Fraud Detection 2023](https://www.kaggle.com/datasets/nelgiriyewithana/credit-card-fraud-detection-dataset-2023) dataset (CC BY 4.0).

To use the full 568K dataset, download from Kaggle and place in `data/raw/`:
```bash
kaggle datasets download -d nelgiriyewithana/credit-card-fraud-detection-dataset-2023 -p data/raw/ --unzip
```

If no real data is available, generate synthetic data instead:

In [ ]:
!python data/generators/generate_all.py --task fraud --num-samples 500
!python data/generators/generate_all.py --task sepsis --num-samples 500

This creates `data/samples/fraud/` and `data/samples/sepsis/` with `data.npz`, `manifest.json`, and `data_card.md`.

All tutorials use pre-generated data from `data/samples/`. To generate all 8 tasks at once:
```bash
for task in fraud sepsis ecg anomaly mortality drug readmission satellite; do
    python data/generators/generate_all.py --task $task --num-samples 500
done
```

## Step 4: Train a Model and Run Inference

Train a simple fraud detection MLP and test it:

In [ ]:
import torch
import torch.nn as nn
import numpy as np

# Load fraud detection data (568K real credit card transactions)
data = np.load("data/samples/fraud/data.npz")
X_all = torch.from_numpy(data["X"])
y_all = torch.from_numpy(data["y"])

print(f"Full dataset: {X_all.shape[0]:,} samples, {X_all.shape[1]} features")
print(f"Fraud rate: {y_all.mean():.2f}")

# Subsample for fast tutorial demo (use full dataset in production)
N = min(5000, len(X_all))
X = X_all[:N]
y = y_all[:N]
print(f"Using first {N:,} samples for training demo")

# Build model (same architecture as models/hfl/mlp/)
model = nn.Sequential(
    nn.Linear(30, 64), nn.ReLU(), nn.Dropout(0.3),
    nn.Linear(64, 64), nn.ReLU(), nn.Dropout(0.3),
    nn.Linear(64, 1), nn.Sigmoid(),
)
print(f"Model: {sum(p.numel() for p in model.parameters()):,} parameters")

# Train
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.BCELoss()

for epoch in range(10):
    model.train()
    pred = model(X).squeeze()
    loss = loss_fn(pred, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    acc = ((pred > 0.5).float() == y).float().mean()
    print(f"  Epoch {epoch+1:2d}: loss={loss.item():.4f}  accuracy={acc.item():.4f}")

# Inference — predict on held-out samples
model.eval()
with torch.no_grad():
    test_samples = X_all[N:N+5]
    predictions = model(test_samples).squeeze()
    true_labels = y_all[N:N+5]
    
    print("\nInference on 5 held-out transactions:")
    for i in range(5):
        pred_label = "FRAUD" if predictions[i] > 0.5 else "legitimate"
        true_label = "FRAUD" if true_labels[i] > 0.5 else "legitimate"
        print(f"  Transaction {i}: predicted={pred_label} (score={predictions[i].item():.4f})  actual={true_label}")

## Step 5: Run the Test Suite

Verify everything works end-to-end:

In [ ]:
# Run strategy tests (fraud, sepsis)
!python tests/run_tests.py fraud
# Run FL smoke test
!python runners/run_ec2.py fraud --synthetic

**Expected:** Both commands complete without errors.

## Step 6: Validate the PET Toolkit

Quick check that all privacy-enhancing technology modules load:

In [ ]:
# Differential Privacy (Opacus)
from fl_pets.dp import compute_epsilon, DP_PRESETS
eps = compute_epsilon(noise_multiplier=1.5, sample_rate=0.01, steps=100)
print(f"DP: epsilon={eps:.4f} at 100 steps (sigma=1.5)")

# Secure Aggregation (Flower)
from fl_pets.secagg import verify_cancellation
import numpy as np
result = verify_cancellation([np.array([1.0, 2.0])], num_clients=3, round_seed=42)
print(f"SecAgg: masks cancel with error {result['max_error']:.2e}")

# Homomorphic Encryption (TenSEAL)
from fl_pets.he import create_context, encrypt, decrypt
ctx = create_context()
enc = encrypt(ctx, [3.14, 2.71])
dec = decrypt(enc + enc)
print(f"HE: encrypted [3.14, 2.71] * 2 = [{dec[0]:.4f}, {dec[1]:.4f}]")

print("\nAll PET modules verified.")

## What Just Happened?

You:
1. **Installed** the FL platform with all dependencies
2. **Generated** synthetic fraud detection data
3. **Trained** a model and **ran inference** on new samples
4. **Tested** the FL pipeline and PET toolkit

This confirms your environment is ready for the remaining tutorials.

## Next Steps

- [Tutorial 2: Your First Model](02-first-model.ipynb) — train a centralised baseline, then compare with FL